## Epistasis for fitness per-generation and per-cycle - all cases

### Parameters of this notebook 

In [ ]:
import pandas as pd           

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import gridspec

from scipy import integrate
from scipy import stats
import random

from bunch import Bunch

In [ ]:
### Update dependent parameters according to input
import os
import os.path
from os import path

## create export directory if necessary
## foldernames for output plots/lists produced in this notebook
import os
FIG_DIR = f'./figures/epistasis/'
os.makedirs(FIG_DIR, exist_ok=True)
print("All  plots will be stored in: \n" + FIG_DIR)

In [ ]:


### execute script to load modules here
exec(open('setup_aesthetics.py').read())


### Test settings

In [ ]:
from selection_coefficient import Problem_M3, get_ODE_solution, plot_solution

In [ ]:
### define solver params
solver_params = {'t_final' : 10, 'timestep' : 10, 'adaptive_timewindow' : True, 'rtol' : 1e-8, 'atol' : 1e-12, 'scoeff_atol':1e-8, 'scoeff_rtol':1e-6}

In [ ]:
strain_params = {'lam':[6.,5.], 'g':[1.,1.], 'Y':[1.,0.5]}

problem = Problem_M3(**strain_params, x =0.4)
sol = get_ODE_solution(problem, **solver_params)

In [ ]:

fig, ax = plt.subplots(figsize = (9,6))
plot_solution(sol,ax)
ax.legend(loc = 'center right')


In [ ]:
from selection_coefficient import estimate_s21, estimate_s21_timecourse

In [ ]:
def sol2relative_abundance(sol):
    
    N1_0, N2_0 = sol.y[0,0], sol.y[1,0]
    
    N1_t, N2_t = sol.y[0,:], sol.y[1,:]
    
    x_t = np.divide(N2_t, np.add(N1_t,N2_t))
    
    return sol.t, x_t

## test
_, x_t = sol2relative_abundance(sol)

In [ ]:
def sol2s_cycle(sol): 
    ## get relative abundance for mutant
    _, x_t = sol2relative_abundance(sol)
    x0, xf = x_t[0], x_t[-1]
    
    s_cycle = np.log(xf/(1-xf)) - np.log(x0/(1-x0))
    return s_cycle
## test
sol2s_cycle(sol)

In [ ]:
def sol2s_gen(sol): 
    s_cycle = sol2s_cycle(sol)
    
    ## get absolute abundances for wild-type
    Nwt_0, _, _ = sol['y'][:,0]
    Nwt_f, _, _ = sol['y'][:,-1]
    
    ## calculate fold-change for wild-type
    LFC_wt = np.log(Nwt_f/Nwt_0)
    
    s_gen = s_cycle/LFC_wt


    return s_gen
## test
sol2s_gen(sol)

### Define list of cases

In [ ]:
## define wild-type traits
lwt, gwt, Ywt = 2., 1., 1.

## convert wild-type to array
wildtype = np.array((lwt,gwt,Ywt))


## define mutations (addivitely on wild-type)
label2mutation = { 'less_lag': (-1,0,0), 'more_lag': (+1,0,0), 'less_growth': (0,-0.2,0), 'more_growth': (0,0.2,0), 
                  'less_yield': (0,0,-0.75), 'more_yield': (0,0,3)}
                  
## convert to arrays
for k,v in label2mutation.items():
        label2mutation[k] = np.array(v)



In [ ]:
## compute all possible cases
list_cases  = []
list_labels = []
for i in label2mutation.keys():
    for j in label2mutation.keys():
        mutant_1 = wildtype + label2mutation[i]
        mutant_2 = wildtype + label2mutation[j]
        mutant_3 = wildtype + label2mutation[i] + label2mutation[j]
        list_cases.append((mutant_1, mutant_2, mutant_3 ))
        list_labels.append(i + '__' + j)

## expect 36 cases
print(len(list_cases))

### Helper functions

In [ ]:
def mutant2strain_params(trait_tuple):
    """Converts tuple of mutant traits into dict of strain parameters.
    
    The expected order of traits in the tuple is 
    0: lag time, 1: growth rate and 2: biomass yield. """
    
    lmut, gmut, Ymut = trait_tuple
    strain_params = {'lam':[lwt, lmut], 'g':[gwt,gmut], 'Y':[Ywt,Ymut]}
    
    return strain_params
    
## test
mutant2strain_params((1,2,3))
    

In [ ]:
def strain_params2result(strain_params):
    
    ### set initial conditions for competition growth cycle
    initial_conditions = {'x':0.01, 'N_0':0.01, 'R_0':0.5}
    
    ## compute ode solution
    problem = Problem_M3(**strain_params, **initial_conditions)
    sol = get_ODE_solution(problem, **solver_params)
    
    ### compute selection coefficient
    s_cycle = sol2s_cycle(sol)
    ### estimate log doubling ratio
    s_gen = sol2s_gen(sol)
    ### estimate mutant relative abundance at the end
    _, x_t = sol2relative_abundance(sol)
    x0,xf = x_t[0],x_t[-1]
    
    ## build title
    title = f'$x_f$={xf:.2f}, ' +r'$s^{\mathrm{logit}}_{\mathrm{cycle}}$'+f'={s_cycle:.3f}, ' + \
    r'$s^{\mathrm{logit}}_{\mathrm{gen}}$'+f'={s_gen:.3f}'
    title = r'$s^{\mathrm{logit}}_{\mathrm{cycle}}$'+f'={s_cycle:.3f}, ' + \
    r'$s^{\mathrm{logit}}_{\mathrm{gen}}$'+f'={s_gen:.3f}'
    
    
    result = Bunch({'sol': sol, 's_gen': s_gen, 's_cycle': s_cycle, 'mut_freq_final': xf , 'title': title})
    
    return result

## test
result = strain_params2result({'lam':[1.,1.], 'g': [1.,1.], 'Y':[1.,1.]})

In [ ]:
### note: the Python print function does not work for Bunch objects


### Compute all cases

In [ ]:

## create datatable to store results
df_stats = pd.DataFrame(index = list_labels)

for label, case in zip(list_labels, list_cases): 

    for j in range(3): 

        ## get competition growth cycle results
        strain_params = mutant2strain_params(case[j])
        result = strain_params2result(strain_params)

        ## store results
        df_stats.at[label, f's_gen_{j+1}'] = result.s_gen
        df_stats.at[label, f's_cycle_{j+1}'] = result.s_cycle
        
## reorder the columns
df_stats = df_stats[df_stats.columns.sort_values()]

## calculate epistasis
df_stats['epistasis_cycle'] = df_stats['s_cycle_3'] - ( df_stats['s_cycle_1'] + df_stats['s_cycle_2'])
df_stats['epistasis_gen']   = df_stats['s_gen_3']   - ( df_stats['s_gen_1'] + df_stats['s_gen_2'])

In [ ]:
def label2fancy(label):
    text = label.replace('__',',\n')
    #remove underscores
    text = text.replace('_',' ')

    ## rewrite the sign as a symbol
    text = text.replace('more',"higher")
    text = text.replace('less',"lower")
    
    ## rewrite the trait as a symbol
    text = text.replace('lag', 'lag time')
    text = text.replace('growth', 'growth rate')
    text = text.replace('yield', 'biomass yield')
    
    return text
## test
label2fancy(label)

In [ ]:
## add fancy labels

for label, case in zip(list_labels, list_cases): 
    
    df_stats.at[label, 'label_fancy'] = label2fancy(label)

    for j in range(3): 

        ## create label
        
        if j == 0:
            text = label.split('__')[0]
        elif j == 1: 
            text = label.split('__')[1]
        else:
            text = label.replace('__',',\n')
         
        #remove underscores
        text = text.replace('_','')
        
        ## rewrite the sign as a symbol
        text = text.replace('more',"+")
        text = text.replace('less',"-")
        ## rewrite the trait as a symbol
        text = text.replace('lag', r"$\lambda$")
        text = text.replace('growth', r"$g$")
        text = text.replace('yield', r"$Y$")
        df_stats.at[label, f'mutation_{j+1}'] =  text

        

In [ ]:
### plot expectation cartoon 

fig, ax = plt.subplots(figsize = (FIGHEIGHT_TRIPLET, FIGHEIGHT_TRIPLET))

sns.scatterplot(x=np.linspace(-5,5,10), y = np.linspace(-5,5,10), ax = ax, s = 80, color = 'black')

## add lines for orientation
ax.axvline(0, ls = '--', color = 'black', zorder = -1)
ax.axhline(0, ls = '--', color = 'black', zorder = -1)

## center on the x-axis
xmin, xmax = ax.get_xlim()
xabsmax = 1.1*np.max(np.abs([xmin,xmax]))
ax.set_xlim(-xabsmax, xabsmax)

## center on the y-axis
ymin, ymax = ax.get_ylim()
yabsmax = 1.1*np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax, yabsmax)

## remove ticks
ax.tick_params(left = False, labelleft = False, bottom = False, labelbottom = False)
## add diagonal line
ax.plot([-xabsmax,xabsmax], [-yabsmax, yabsmax], color = 'tab:grey', ls = 'dotted')

## set labels
ax.set_ylabel("epistasis value\nfitness per-generation: $s_\mathrm{gen}$")
ax.set_xlabel("epistasis value\nfitness per-cycle: $s_\mathrm{cycle}$")

fig.savefig(FIG_DIR + f"cartoon_epistasis_per-cycle_vs_epistasis_per-gen_perfect_correlation.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
rcParams['axes.labelsize'] = 15  # fontsize of the x and y labels
rcParams['ytick.labelsize'] = 15    # fontsize of the tick labels

sns.set_theme(style = 'ticks',font_scale = 1.5,rc=rcParams)

In [ ]:
fig, ax = plt.subplots(figsize = (0.8*FIGHEIGHT_TRIPLET, 0.8*FIGHEIGHT_TRIPLET))

ax = sns.scatterplot(df_stats, x='epistasis_gen', y='epistasis_cycle', ax = ax, s = 80, color = 'black')

## add lines for orientation
ax.axvline(0, ls = '--', color = 'black', zorder = -1)
ax.axhline(0, ls = '--', color = 'black', zorder = -1)

## center on the x-axis
xmin, xmax = ax.get_xlim()
xabsmax = 1.1*np.max(np.abs([xmin,xmax]))
ax.set_xlim(-xabsmax, xabsmax)

## center on the y-axis
ymin, ymax = ax.get_ylim()
yabsmax = 1.1*np.max(np.abs([ymin,ymax]))
ax.set_ylim(-yabsmax, yabsmax)

## add diagonal line
ax.plot([-xabsmax,xabsmax], [-yabsmax, yabsmax], color = 'tab:grey', ls = 'dotted')

## set labels
#ax.set_ylabel("epistasis value\nfitness per-generation: $s_\mathrm{gen}$")
ax.set_xlabel("epistasis (per-generation)")
#ax.set_xlabel("epistasis value\nfitness per-cycle: $s_\mathrm{cycle}$")
ax.set_ylabel("epistasis (per-cycle)")

fig.savefig(FIG_DIR + f"correlation_epistasis_per-cycle_vs_epistasis_per-gen.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Plot selection coefficients as heatmap

In [ ]:
def index2sort_order(index): 
    labels = np.array(index)
    a = np.array([1 if '+' in v else 2 for v in labels])
    b = np.array([1 if 'g' in v else 10 if 'lambda' in v else 100 for v in labels])

    return np.multiply(a,b)
## test
index2sort_order(df_stats.index)

In [ ]:
def variable2data(var = 'epistasis_cycle'): 
    ## reshape data
    data = df_stats[['mutation_1', 'mutation_2', var]]
    data = data.pivot(index = 'mutation_1', columns = 'mutation_2',)
    data.columns = data.columns.droplevel()
    
    data = data.sort_index(axis = 0, key=index2sort_order)
    data = data.sort_index(axis = 1, key=index2sort_order)
    
    return data

## test
variable2data(var = 'epistasis_cycle')


In [ ]:
import matplotlib

### Plot epistasis heatmaps

In [ ]:
## increase font-sizes


rcParams = dict()
rcParams['axes.labelsize'] = 18    # fontsize of the x and y labels
rcParams['ytick.labelsize'] = 16    # fontsize of the tick labels

sns.set_theme(style = 'ticks',font_scale = 1.5,rc=rcParams)


In [ ]:
## define variable
var = 'epistasis_gen'
data = variable2data(var)

## get range of data values
value_max = yabsmax #data.max(axis =1).max(axis=0)
value_min = xabsmax #data.min(axis =1).min(axis=0)
abs_max = np.max([np.abs(value_min), np.abs(value_max)])
## define normalization for colormap
norm = matplotlib.colors.Normalize(vmin = -abs_max, vmax = abs_max)
## choose colormap
cmap = sns.diverging_palette(220, 20, as_cmap=True)

In [ ]:
fig, ax = plt.subplots(figsize = (FIGWIDTH_TRIPLET, FIGHEIGHT_TRIPLET))
cbar_kws={'label': 'epistasis value'} #\n fitness per-generation: $s_\mathrm{gen}$'}

sns.heatmap(data.T,cmap=cmap, linewidth=.5, cbar_kws=cbar_kws, norm = norm,  ax = ax) 
ax.tick_params(labelsize = 19)
#ax.set_xlabel("effect of mutation 1")
ax.set_xlabel("")
#ax.set_ylabel("effect of mutation 2")
ax.set_ylabel("")
# Set ticks on both sides of axes on
#ax.tick_params(axis="x", bottom=True, top=True, labelbottom=True, labeltop=True)
fig.savefig(FIG_DIR + f"heatmap_epistasis_per-generation.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
## define variable
var = 'epistasis_cycle'
data = variable2data(var)


In [ ]:
## get range of data values
value_max = data.max(axis =1).max(axis=0)
value_min = data.min(axis =1).min(axis=0)
assert np.abs(value_min) < abs_max, "Need to increase colormap range."
assert np.abs(value_max) < abs_max, "Need to increase colormap range."
## we use same norm and colormap as for selection coefficient per-cycle


In [ ]:
fig, ax = plt.subplots(figsize = (FIGWIDTH_TRIPLET, FIGHEIGHT_TRIPLET))
cbar_kws={'label': 'epistasis value'} #\n fitness per-generation: $s_\mathrm{gen}$'}
#plt.xticks(fontname = "Times New Roman")

sns.heatmap(data.T,cmap=cmap, linewidth=.5, cbar_kws=cbar_kws, norm = norm,  ax = ax) 
ax.tick_params(labelsize = 19)
#ax.set_xlabel("effect of mutation 1")
ax.set_xlabel("")
#ax.set_ylabel("effect of mutation 2")
ax.set_ylabel("")

# Set ticks on both sides of axes on
#ax.tick_params(axis="x", bottom=True, top=True, labelbottom=True, labeltop=True)
fig.savefig(FIG_DIR + f"heatmap_epistasis_per-cycle.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
ax.get_xticklabels()

In [ ]:
ax.get_yticks()

### Plot fitness heatmaps

In [ ]:
## define variable
var = 's_cycle_3'
data = variable2data(var)

## get range of data values
value_max = data.max(axis =1).max(axis=0)
value_min = data.min(axis =1).min(axis=0)
abs_max = np.max([np.abs(value_min), np.abs(value_max)])
## define normalization for colormap
norm = matplotlib.colors.Normalize(vmin = -abs_max, vmax = abs_max)
## choose colormap
cmap = sns.diverging_palette(220, 20, as_cmap=True)

In [ ]:
fig, ax = plt.subplots(figsize = (FIGWIDTH_TRIPLET, FIGHEIGHT_TRIPLET))
cbar_kws={'label': 'double mutant\n fitness per-cycle: $s_\mathrm{cycle}$'}

sns.heatmap(data.T,cmap=cmap, linewidth=.5, cbar_kws=cbar_kws, norm = norm,  ax = ax) 
ax.tick_params(labelsize = BIGGER_SIZE)
#ax.set_xlabel("")
#ax.set_ylabel("")

# Set ticks on both sides of axes on
#ax.tick_params(axis="x", bottom=True, top=True, labelbottom=True, labeltop=True)
fig.savefig(FIG_DIR + f"heatmap_fitness_double_mutant_per-cycle.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
## define variable
var = 's_gen_3'
data = variable2data(var)


In [ ]:
## get range of data values
value_max = data.max(axis =1).max(axis=0)
value_min = data.min(axis =1).min(axis=0)
assert np.abs(value_min) < abs_max, "Need to increase colormap range."
assert np.abs(value_max) < abs_max, "Need to increase colormap range."
## we use same norm and colormap as for selection coefficient per-cycle


In [ ]:
fig, ax = plt.subplots(figsize = (FIGWIDTH_TRIPLET, FIGHEIGHT_TRIPLET))
cbar_kws={'label': 'fitness per-generation: $s_\mathrm{gen}$'}

sns.heatmap(data.T,cmap=cmap, linewidth=.5, cbar_kws=cbar_kws, norm = norm,  ax = ax) 
ax.tick_params(labelsize = BIGGER_SIZE)
ax.set_xlabel("")
ax.set_ylabel("")

# Set ticks on both sides of axes on
#ax.tick_params(axis="x", bottom=True, top=True, labelbottom=True, labeltop=True)
fig.savefig(FIG_DIR + f"heatmap_fitness_double_mutant_per-generation.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

### Choose plot parameters

In [ ]:
### choose linewidth
lw = 3

### choose wildtype color
color_wt = 'grey'

### choose default mutant color
color_mut = ('tab:blue', 'tab:red', 'tab:purple')

In [ ]:
### set boundary for graphics
t_final = 10
ymax = 10
ymin = 0.001



### Example: plot for a single case

In [ ]:
i = 3
case  = list_cases[i]
label = list_labels[i]
print(label)

In [ ]:
def label2growth_curves_plot(label, case): 
    ## define figure layout
    pad = 0.4
    gridspec_kw = {"width_ratios":[1,1,1], "wspace":0.15, 'hspace':0.0}
    growthcurve_params = {'nrows': 1, 'ncols': 3,  
                          'figsize': (3*0.6*FIGWIDTH_TRIPLET, 0.7*FIGHEIGHT_TRIPLET),
                          #'figsize': (3*0.8*FIGWIDTH_TRIPLET+3*pad, 0.7*FIGHEIGHT_TRIPLET), 
                      'gridspec_kw': gridspec_kw }
    
    fig , axes = plt.subplots(**growthcurve_params, sharey = True)

    for i in range(3): 
        ## plot mutant one 
        ax = axes[i]
        ## get competition growth cycle results
        strain_params = mutant2strain_params(case[i])
        result = strain_params2result(strain_params)

        ## plot wildtype
        ax.plot(result.sol.t, result.sol.y[0], color = color_wt, lw = lw, label = 'wildtype $N_1$')
        ## plot mutant
        ax.plot(result.sol.t, result.sol.y[1], color = color_mut[i], lw = lw, label = 'lag-time mutant $N_3$')
        ## set ttile
        ax.set_title(result.title, loc = 'center', y = 0.8)

    for ax in axes:
        ax.set_xlim(0,t_final)
        ax.set_ylim(ymin,ymax)
        ax.set_yscale('log')

        #ax.legend(loc = 'lower right')

        ax.set_xlabel('time')
        
    axes[0].set_ylabel('absolute abundance')
        
    return fig

## test
fig = label2growth_curves_plot(label, case)
fig.savefig(FIG_DIR + f"growthcurves_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)

In [ ]:
def add_epistasis(s1,s2,s3, ax): 
    """Adds lines and dots for three selection coefficients."""
    
    ## plot lines (looks like parallelogram with no epistasis)
    color_lines = 'tab:grey'
    ax.plot([0,1], [0,s1], color = color_lines,  ls = '--')
    ax.plot([1,2], [s1,s3], color = color_lines, zorder = -1, ls = '--')
    ax.plot([0,1], [0,s2], color = color_lines, ls = '--')
    ax.plot([1,2], [s2,s3], color = color_lines, zorder = -1, ls = '--')
    
    # plot dots
    ax.scatter([0], [0], color = color_wt, marker = 'o', zorder = 10, s =80)
    ax.scatter([1], [s1], color = color_mut[0], marker = 'o', zorder = 10, s =80)
    ax.scatter([1], [s2], color = color_mut[1], marker = 'o', zorder = 10, s =80)
    ax.scatter([2], [s3], color = color_mut[2], marker = 'o', zorder = 10, s =80)

    
    return ax

In [ ]:
def label2epistasis_plot(label): 
    
    ## define new plot layout
    pad = 0.25
    gridspec_kw = {"width_ratios":[1,1], "wspace":pad, 'hspace':0.0}

    epistasis_plot_params = {'nrows': 1, 'ncols': 2, 'figsize' : (2*0.6*FIGWIDTH_TRIPLET+pad, 0.7*FIGHEIGHT_TRIPLET), 
                             'gridspec_kw': gridspec_kw}
    ### plot epistasis diagrams
    fig , axes = plt.subplots(**epistasis_plot_params)

    ax = axes[0]

    ## retrieve selection coeffcients
    s1,s2,s3 = df_stats.loc[label, ['s_gen_1', 's_gen_2', 's_gen_3']]
    ax = add_epistasis(s1,s2,s3, ax)

    ## add epistasis value to title
    epis = df_stats.at[label, 'epistasis_gen']
    title = f"epistasis value: {epis:.3f}"
    ax.set_title(title, loc = 'right', fontsize = BIGGER_SIZE)

    ## fix axis labels
    ax.set_ylabel('fitness per-generation: '+ r'$s^{\mathrm{logit}}_{\mathrm{gen}}$')

    ax = axes[1]

    ## retrieve selection coeffcients
    s1,s2,s3 = df_stats.loc[label, ['s_cycle_1', 's_cycle_2', 's_cycle_3']]
    ax = add_epistasis(s1,s2,s3, ax)

    ## add epistasis value to title
    epis = df_stats.at[label, 'epistasis_cycle']
    title = f"epistasis value: {epis:.3f}"
    ax.set_title(title, loc = 'right', fontsize = BIGGER_SIZE)

    ## fix axis labels
    ax.set_ylabel('fitness per-cycle: '+ r'$s^{\mathrm{logit}}_{\mathrm{cycle}}$')

    for ax in axes: 
        ax.set_xticks([0,1,2])
        ax.set_xticklabels(['wild\n-type', 'single\nmutants', 'double\nmutant'])
        ax.tick_params(left= False, labelleft = False)
        
    return fig

## test

fig = label2epistasis_plot(label)
fig.savefig(FIG_DIR + f"epistasis_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)
    

In [ ]:
### get axis limits

lmin, lmax = lwt,lwt
gmin, gmax = gwt,gwt
Ymin, Ymax = Ywt,Ywt

for case in list_cases: 
    
    for i in range(3): 
        l,g,Y = case[i]

        if l < lmin: lmin = l
        elif l > lmax: lmax = l


        if g < gmin: gmin = g
        elif g > gmax: gmax = g


        if Y < Ymin: Ymin = Y
        elif Y> Ymax: Ymax = Y

print((lmin,lmax))
print((gmin,gmax))
print((Ymin,Ymax))

In [ ]:
def label2trait_histogram(label, case): 
    ### add trait histogram

    ## define figure layout
    pad = 0.4
    gridspec_kw = {"height_ratios": [1, 1, 1], "wspace":0, 'hspace':0.1}
    traitplot_params = {'nrows': 3, 'ncols': 1, 'figsize': (0.5*FIGWIDTH_TRIPLET, 0.7*FIGHEIGHT_TRIPLET), 
                      'gridspec_kw': gridspec_kw }

    fig, axes = plt.subplots(**traitplot_params,sharex = True)

    barplot_params = {'width': 0.5, 'color': [color_wt, *color_mut]}

    ## plot lag times
    ax = axes[0]
    l1,l2,l3 = (case[j][0] for j in range(3))
    ax.bar([0,1,2,3], [lwt, l1,l2,l3] , **barplot_params)
    ax.set_ylabel("lag\ntime",rotation = 0, va = 'center', rotation_mode = 'default', ma = 'right', ha = 'right')
    ax.set_ylim(0,1.05*lmax)

    ## plot growth rates
    ax = axes[1]
    g1,g2,g3 = (case[j][1] for j in range(3))
    ax.bar([0,1,2,3], [gwt, g1,g2,g3] , **barplot_params)
    ax.set_ylabel("growth\nrate",rotation = 0, va = 'center', rotation_mode = 'default', ma = 'right', ha = 'right')
    ax.set_ylim(0, 1.05*gmax)

    ## plot biomass yield
    ax = axes[2]
    Y1,Y2,Y3 = (case[j][2] for j in range(3))
    ax.bar([0,1,2,3], [Ywt, Y1,Y2,Y3] , **barplot_params)
    ax.set_ylabel("biomass\nyield",rotation = 0, va = 'center', rotation_mode = 'default', ma = 'right', ha = 'right')
    ax.set_ylim(0, 1.05*Ymax)
    
    for ax in axes: 
        ax.tick_params(left = False, labelleft = False)

    ax.set_xticks([0,1,2, 3])
    ax.set_xticklabels(['wild-\ntype', 'mutant\n1', 'mutant\n2','double\nmutant'])
    
    return fig

## test
fig = label2trait_histogram(label, case)
fig.savefig(FIG_DIR + f"traits_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)
 

### Plot for all cases

In [ ]:
PAD_INCHES = 0.2

In [ ]:
for label, case in zip(list_labels, list_cases): 
    
    ## plot growth curves
    fig = label2growth_curves_plot(label, case)
    fig.savefig(FIG_DIR +  f"growthcurves_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)
    plt.close(fig)
    
    ## plot traits
    fig = label2trait_histogram(label, case)
    fig.savefig(FIG_DIR + f"traits_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)
    plt.close(fig)
    
    ## plot epistasis   
    fig = label2epistasis_plot(label)
    fig.savefig(FIG_DIR + f"epistasis_case_{label}.pdf", DPI = DPI, bbox_inches = 'tight', pad_inches = PAD_INCHES)
    plt.close(fig)